In [7]:
import pandas as pd

print("Loading 2019...")
df_2019 = pd.read_csv("../data/seoul-metro-2019.logs.csv")
df_2019["year"] = 2019
print(f"2019 loaded: {len(df_2019):,} rows")

print("\nLoading 2021...")
df_2021 = pd.read_csv("../data/seoul-metro-2021.logs.csv")
df_2021["year"] = 2021
print(f"2021 loaded: {len(df_2021):,} rows")

Loading 2019...
2019 loaded: 2,008,040 rows

Loading 2021...
2021 loaded: 1,048,575 rows


In [8]:
# ── Combine ──────────────────────────────────────────
df = pd.concat([df_2019, df_2021], ignore_index=True)
print(f"Combined total rows: {len(df):,}")
print(f"Columns: {df.columns.tolist()}")

Combined total rows: 3,056,615
Columns: ['timestamp', 'station_code', 'people_in', 'people_out', 'year']


In [9]:
# ── Fix timestamp ────────────────────────────────────
df['timestamp'] = pd.to_datetime(df['timestamp'], utc=True)
df['timestamp'] = df['timestamp'].dt.tz_localize(None)

# ── Sort by station and time ─────────────────────────
df = df.sort_values(['station_code', 'timestamp'])
df = df.reset_index(drop=True)

print("Sorted successfully")
print("\nDate range:")
print("Start:", df['timestamp'].min())
print("End:  ", df['timestamp'].max())

Sorted successfully

Date range:
Start: 2018-12-31 20:00:00
End:   2021-07-17 14:00:00


In [10]:
# ── Check quality ────────────────────────────────────
print("Null values:")
print(df.isnull().sum())

print(f"\nTotal stations: {df['station_code'].nunique()}")

dupes = df.duplicated(
    subset=['timestamp','station_code']).sum()
print(f"Duplicate rows: {dupes}")

if dupes > 0:
    df = df.drop_duplicates(
        subset=['timestamp','station_code'])
    print(f"Duplicates removed. Remaining: {len(df):,}")

Null values:
timestamp       0
station_code    0
people_in       0
people_out      0
year            0
dtype: int64

Total stations: 283
Duplicate rows: 0


In [11]:
# ── Save combined file ───────────────────────────────
output = '../data/combined_2019_2021.logs.csv'
df.to_csv(output, index=False)

size = os.path.getsize(output) / (1024*1024)
print(f"Saved: {output}")
print(f"File size: {size:.1f} MB")
print(f"Total rows: {len(df):,}")
print("\nFirst 3 rows:")
print(df.head(3))

Saved: ../data/combined_2019_2021.logs.csv
File size: 113.1 MB
Total rows: 3,056,615

First 3 rows:
            timestamp  station_code  people_in  people_out  year
0 2018-12-31 20:00:00           150        348         222  2019
1 2018-12-31 21:00:00           150        321         821  2019
2 2018-12-31 22:00:00           150        348         808  2019
